In [88]:
import numpy as np
import pandas as pd

In [4]:
# read in dataset
df_hs = pd.read_csv("data_use.csv")
df_hs

,user.index,date,time,5min_datetime,steps,rolling_sum,rt,sugg.decision.5min,send.sedentary,availability
0,1,7/22/2015,9:05:00,NaN,0,0,1,NaN,0,1
1,1,7/22/2015,9:10:00,NaN,0,0,1,NaN,0,1
2,1,7/22/2015,9:15:00,NaN,0,0,1,NaN,0,1
3,1,7/22/2015,9:20:00,NaN,0,0,1,NaN,0,1
4,1,7/22/2015,9:25:00,NaN,0,0,1,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...
228235,37,1/26/2016,20:40:00,NaN,0,0,1,NaN,0,1
228236,37,1/26/2016,20:45:00,NaN,0,0,1,NaN,0,1
228237,37,1/26/2016,20:50:00,NaN,0,0,1,NaN,0,1
228238,37,1/26/2016,20:55:00,NaN,0,0,1,NaN,0,1


In [ ]:
# create subset of the data to work on


## Define `transform_dnn`

In [107]:
def transform_dnn(df, day_partitions = 4, day_measure = 144, units = 37):
    """
    Transfom df into shape suitable for DNN use case.

    Parameters
    ----------
    df : pandas DataFrame
         a matrix of size Z x 10 (with a set of properties to be defined)
         ...
    day_partitions : int
        The number of partitions to split an individual day into. The 
        default value is 4. Maximally, the value of day_partitions is 
        day_measure.
    day_measure : int
        The number of measurements per "day" in df. The default is 144,
        which aligns with the measurements per day in the HeartSteps dataset.
    units : int
        The number of units in df. Default is 37 (in line with the HeartSteps dataset)        
    Returns
    ----------
    tuple (M, A)
    where M is a units x (day_partitions * D) x (day_measure / day_partitions) np array 
    that contains the outcome vectors of interest and A is a np array shape units x (day_partitions * D)
    that indicates in which unit/time combination a treatment was observed.
    D is the maximum length of days observed among all unit trials.
    """
    # first, find the maximum trial length in days
    # units are 1-indexed
    max_days = 0
    num_days = []
    for i in range(1, units + 1):
        len_i = len(df[df['user.index'] == i]) // day_measure
        num_days.append(len_i)
        if len_i > max_days:
            max_days = len_i
    # init M and A
    M = np.zeros((units, day_partitions * max_days, day_measure // day_partitions))
    A = np.zeros((units, day_partitions * max_days))
    
    for i in range(units):
        # note: i is used to index into M and A.
        # unit_ind is used to retrieve unit specific values from df
        unit_ind = i + 1
        
        # slice by unit
        df_unit = df.loc[df["user.index"] == unit_ind]
        
        # pad first to max trial length to maintain partition size across units
        trial_steps = df_unit["steps"].to_numpy()
        trial_trt = df_unit["treatment"].to_numpy()
        if num_days[i] < max_days:
            # pad with 0s, step values are masked by trt
            pad = np.zeros((max_days - num_days[i]) * day_measure)
            trial_steps = np.append(trial_steps, pad)
            pad_trt = np.zeros((max_days - num_days[i]) * day_measure)
            trial_trt = np.append(trial_trt, pad_trt)
            
        # first split into days, then into partitions
        lst_days = np.split(trial_steps, max_days)
        parts = np.array([np.split(x, day_partitions) for x in lst_days])
         
        # Shape of parts is max_days x day_partitions x (day_measure // day_partitions)
        # Flatten along d dimension to achieve (day_partitions * max_days) x (day_measure // day_partitions) shape
        unit_parts = parts.reshape((max_days * day_partitions, day_measure // day_partitions))
        
        # Repeat for treatment col. 
        lst_days_trt = np.split(trial_trt, max_days)
        parts_trt = np.array([np.split(x, day_partitions) for x in lst_days_trt])
        unit_trt = parts_trt.reshape((max_days * day_partitions, day_measure // day_partitions))
        
        # if any of the trt values within the partition is 1, set the value in the final array to 1
        unit_trt_agg = unit_trt.any(axis = 1).astype(int)
        
        M[i] = unit_parts
        A[i] = unit_trt_agg
    return M, A
    

## Testing `transform_dnn` with simulated symmetrical data

We create a small test dataset as a proof of concept of the above data transformation. This data set will have 3 units over a time span of 4 days, with 12 measurements a day. We partition each day into 3 segments (4 measurements per partition). The treatment is randomly applied across each day and time span. This is a simpler example than the actual use case, as the number of days within the trial is consistent for all users. See the section below for a more complex example.

In [90]:
# create test set

# user inds
u1 = np.repeat(1, 12 * 4)
u2 = np.repeat(2, 12 * 4)
u3 = np.repeat(3, 12 * 4)
user_index = np.concatenate((u1, u2, u3))

# step measurements (for simplicity, between 1 and 10)
rng = np.random.default_rng()
u1_steps = rng.integers(low=1, high=10, size=12 * 4)
u2_steps = rng.integers(low=1, high=10, size=12 * 4)
u3_steps = rng.integers(low=0, high=10, size=12 * 4)
steps = np.concatenate((u1_steps, u2_steps, u3_steps))
 
# treatment measurements
# in practice, it seems much more likely to see no treatment rather than a treatment
u1_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
u2_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
u3_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
trt = np.concatenate((u1_trt, u2_trt, u3_trt))

# create final dataframe
final_np = np.stack((user_index, steps, trt), axis = 1)
df_symtest = pd.DataFrame(final_np, columns = ["user.index", "steps", "treatment"])
df_symtest

,user.index,steps,treatment
0,1,2,0
1,1,4,0
2,1,7,0
3,1,2,0
4,1,1,0
...,...,...,...
139,3,4,0
140,3,0,0
141,3,8,0
142,3,3,0


Now that we have created our test set, we can use `transform_dnn` to demonstrate the desired transformation properties.

In [91]:
M, A = transform_dnn(df_symtest, day_partitions = 3, day_measure = 12, units = 3)

In [92]:
M

array([[[2., 4., 7., 2.],
        [1., 9., 8., 8.],
        [5., 6., 8., 4.],
        [4., 8., 3., 9.],
        [9., 7., 7., 4.],
        [1., 4., 9., 2.],
        [4., 8., 5., 7.],
        [7., 5., 9., 4.],
        [5., 9., 6., 6.],
        [6., 7., 4., 2.],
        [8., 2., 2., 4.],
        [5., 7., 6., 8.]],

       [[6., 3., 8., 5.],
        [9., 4., 8., 4.],
        [5., 7., 3., 9.],
        [5., 2., 7., 1.],
        [5., 6., 2., 9.],
        [5., 6., 5., 5.],
        [3., 7., 5., 2.],
        [1., 7., 3., 1.],
        [7., 5., 9., 6.],
        [7., 7., 7., 5.],
        [3., 1., 8., 1.],
        [1., 9., 5., 2.]],

       [[3., 3., 9., 9.],
        [6., 8., 0., 0.],
        [0., 7., 5., 5.],
        [3., 3., 5., 3.],
        [2., 9., 2., 9.],
        [3., 2., 8., 2.],
        [4., 4., 9., 8.],
        [3., 1., 4., 9.],
        [7., 6., 5., 1.],
        [7., 2., 5., 3.],
        [6., 2., 6., 4.],
        [0., 8., 3., 7.]]])

In [93]:
A

array([[0., 0., 0., 1., 0., 0., 1., 1., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0., 1., 1., 1., 1., 1., 1., 0.],
       [1., 1., 1., 0., 1., 0., 0., 1., 0., 1., 0., 0.]])

## Testing `transform_dnn` on simulated asymmetrical data

Next, we test the function on a dataset with asymmetrical trial periods. As before, we will have 3 units, but this time, one of the units has a longer 6 day trial period. The remaining units have a 4 day trial period. There are still 12 measurements a day, and each day has 3 segments.

In [105]:
# create test set
# user inds, u1 has 6 day trial period
u1 = np.repeat(1, 12 * 6)
u2 = np.repeat(2, 12 * 4)
u3 = np.repeat(3, 12 * 4)
user_index = np.concatenate((u1, u2, u3))

# step measurements (for simplicity, between 1 and 10)
rng = np.random.default_rng()
u1_steps = rng.integers(low=1, high=10, size=12 * 6)
u2_steps = rng.integers(low=1, high=10, size=12 * 4)
u3_steps = rng.integers(low=0, high=10, size=12 * 4)
steps = np.concatenate((u1_steps, u2_steps, u3_steps))
 
# treatment measurements
# in practice, it seems much more likely to see no treatment rather than a treatment
u1_trt = np.random.choice([0, 1], 12 * 6, p=[0.8, 0.2])
u2_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
u3_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
trt = np.concatenate((u1_trt, u2_trt, u3_trt))

# create final dataframe
final_np = np.stack((user_index, steps, trt), axis = 1)
df_asymtest = pd.DataFrame(final_np, columns = ["user.index", "steps", "treatment"])
df_asymtest

,user.index,steps,treatment
0,1,6,0
1,1,2,0
2,1,6,1
3,1,6,0
4,1,4,1
...,...,...,...
163,3,9,0
164,3,1,0
165,3,3,0
166,3,6,0


In [108]:
M_a, A_a = transform_dnn(df_asymtest, day_partitions = 3, day_measure = 12, units = 3)

In [109]:
M_a

array([[[6., 2., 6., 6.],
        [4., 5., 4., 2.],
        [3., 1., 7., 7.],
        [5., 2., 9., 7.],
        [4., 2., 6., 2.],
        [8., 4., 4., 3.],
        [7., 9., 9., 4.],
        [1., 6., 3., 8.],
        [4., 5., 7., 1.],
        [2., 1., 6., 7.],
        [7., 6., 9., 1.],
        [3., 9., 1., 8.],
        [5., 1., 2., 6.],
        [8., 3., 2., 6.],
        [5., 8., 3., 1.],
        [9., 6., 2., 4.],
        [7., 6., 8., 3.],
        [5., 2., 2., 9.]],

       [[6., 2., 5., 2.],
        [2., 7., 2., 3.],
        [7., 4., 8., 5.],
        [3., 6., 3., 5.],
        [3., 8., 9., 2.],
        [5., 6., 4., 1.],
        [8., 5., 4., 3.],
        [9., 1., 1., 8.],
        [4., 9., 6., 5.],
        [1., 1., 4., 1.],
        [2., 6., 3., 7.],
        [9., 7., 3., 4.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]],

       [[6., 9., 1., 2.],
        [2., 2., 8., 3.],
        

In [110]:
A_a

array([[1., 1., 1., 1., 1., 0., 1., 0., 0., 1., 0., 1., 1., 1., 0., 1.,
        1., 1.],
       [0., 1., 0., 0., 0., 0., 1., 0., 1., 0., 0., 1., 0., 0., 0., 0.,
        0., 0.],
       [1., 0., 1., 0., 0., 1., 0., 0., 1., 1., 0., 0., 0., 0., 0., 0.,
        0., 0.]])

## Test `transform_dnn` on HeartSteps dataset

Finally, we test `transform_dnn` on its primary use case, the HeartSteps dataset. We use `send.sedentary` as the "treatment" indicator.

In [113]:
df_hs["treatment"] = df_hs["send.sedentary"]

,user.index,date,time,5min_datetime,steps,rolling_sum,rt,sugg.decision.5min,send.sedentary,availability,treatment
0,1,7/22/2015,9:05:00,NaN,0,0,1,NaN,0,1,0
1,1,7/22/2015,9:10:00,NaN,0,0,1,NaN,0,1,0
2,1,7/22/2015,9:15:00,NaN,0,0,1,NaN,0,1,0
3,1,7/22/2015,9:20:00,NaN,0,0,1,NaN,0,1,0
4,1,7/22/2015,9:25:00,NaN,0,0,1,NaN,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...
228235,37,1/26/2016,20:40:00,NaN,0,0,1,NaN,0,1,0
228236,37,1/26/2016,20:45:00,NaN,0,0,1,NaN,0,1,0
228237,37,1/26/2016,20:50:00,NaN,0,0,1,NaN,0,1,0
228238,37,1/26/2016,20:55:00,NaN,0,0,1,NaN,0,1,0


We can see that there are 37 units. Time statistics run from 9:05 to 21:00 at 5 minute intervals, yielding 12 observations per hour for 12 hours. This means that there are 144 measurements per day. We will demonstrate by partitioning the data into 36 sections per day (i.e. each section has 4 measurements). 

In [268]:
M_hs, A_hs = transform_dnn(df_hs, day_partitions = 4, day_measure = 144, units = 37)

In [262]:
len(M_hs[np.nonzero(M_hs)])

72705

In [269]:
A_hs = A_hs.astype('int64')

## Li 2019 Nearest Neighbors Implementation

We implement the user-user nearest neighbors algorithm as defined in Nearest Neighbors for Matrix Estimation Interpreted as Blind Regression for Latent Variable Model (Li et al. 2019). More specifically, we construct Algorithm 2, which can be found in Section III [here](https://arxiv.org/abs/1705.04867).

First, to align the NN implementation with the data format considered above, we collapse the partitioned measurements dimension (3rd dim) by taking the mean.

In [134]:
def mean_collapse(A, d = 2):
    """
    Returns A (3d array) with last dimension collapsed as mean
    """
    return np.mean(A, axis = d)

In [ ]:
# do we allow row u to be its own neighbor in the case that we are denoising entry A(u, i)?
# do I use the dissim shown in Algorithm 2 or 4?

In [284]:
def dissim_row(O, M, u, v):
    """
    Returns the row dissimilarity between rows u and v
    in O
    
    Parameters
    ----------
    O : a matrix of size a x b
    M : a masking matrix of size a x b that
        dictates which entries of O are known 
    u : the index of a row in O
    v : the index of a row in O

    Returns
    ----------
    d_row : the row dissimilarity between u and v (float)
    """
    obv_col_mask = M[u] * M[v]
    overlap = np.sum(obv_col_mask)
    if not overlap:
        return float('inf')    
    se = (O[u] - O[v])**2 * obv_col_mask 
    mse = (1 / overlap) * np.sum(se)
    return mse

def nearest_neighbors(O_1, u, M, eta):
    """
    Returns the B neighborbood using O_1 within
    radius eta. 
    
    Parameters
    ----------
    O_1 : a matrix of size c x d
    M : a masking matrix of size c x d that
        dictates which entries of O_1 are known 
    eta : radius of B neighborhood
    
    Returns
    ----------
    B : neighborhood (indicies of most similar rows)
        with radius eta
    """
    dissims = np.array([dissim_row(O_1, M, u, v) for v in range(len(O_1))])
    B = np.nonzero(dissims <= eta)
    return B
    
    
    
def estimate_li_nn(Z, M, eta):
    """
    Returns A_hat, the estimate of partially unobvserved
    matrix A using Nearest Neighbors imputation
    
    Parameters
    ----------
    Z : an array of size m x n x q
    M : a masking matrix of size m x n that
        dictates which entries of Z are known 
    eta : radius of B neighborhood
    
    Returns
    ----------
    A_hat : the estimate of partially unobvserved matrix A
    """
    Z = mean_collapse(Z, d = 2)
    m, n = Z.shape
    A_hat = np.zeros((m, n))
    for u in range(m):
        for v in range(n):
            O_1  = np.delete(Z, v, axis = 1)
            O_2 = Z[:, v]
            M_v = np.delete(M, v, axis = 1)
            u_neighbors = nearest_neighbors(O_1, u, M_v, eta)
            # only select obvserved neighbors
            mask = M[u_neighbors, v]
            pos_nn = Z[u_neighbors, v]
            masked_nn = pos_nn[mask == 1]
            if not len(masked_nn):
                A_hat[u, v] = 0
            else:
                A_hat[u, v] = np.mean(masked_nn)
    return A_hat

def estimate_li_nn_inds(Z, M, eta, v_inds):
    """
    Returns A_hat, the estimate of partially unobvserved
    matrix A using Nearest Neighbors imputation, with only indices specified
    by v_inds estimated
    
    Parameters
    ----------
    Z : an array of size m x n x q
    M : a masking matrix of size m x n that
        dictates which entries of Z are known 
    eta : radius of B neighborhood
    v_inds : indices to estimate, tuple of (x_coords, y_coords)
    
    Returns
    ----------
    A_hat : the estimate of values in partially unobserved matrix A specified by v_inds
    """
    row_coords, col_coords = v_inds
    Z = mean_collapse(Z, d = 2)
    m, n = Z.shape
    A_hat = np.zeros((m, n))
    for i in range(len(row_coords)):
        u = row_coords[i]
        j = col_coords[i]
        O_1  = np.delete(Z, j, axis = 1)
        O_2 = Z[:, j]
        M_j = np.delete(M, j, axis = 1)
        u_neighbors = nearest_neighbors(O_1, u, M_j, eta)
        mask = M[u_neighbors, j]
        pos_nn = Z[u_neighbors, j]
        masked_nn = pos_nn[mask == 1]
        if not len(masked_nn):
            A_hat[u, j] = 0
        else:
            A_hat[u, j] = np.mean(masked_nn)
    return A_hat
        
        
#     for i in range(len(x_coords)):
#         for j in range(len(y_coords)):
#             O_1  = np.delete(Z, j, axis = 1)
#             O_2 = Z[:, j]
#             M_v = np.delete(M, j, axis = 1)
#             i_neighbors = nearest_neighbors(O_1, i, M_v, eta)
#             # only select obvserved neighbors
#             mask = M[i_neighbors, j]
#             pos_nn = Z[i_neighbors, j]
#             masked_nn = pos_nn[mask]
#             if not len(masked_nn):
#                 A_hat[i, j] = 0
#             else:
#                 A_hat[i, j] = np.mean(masked_nn)
#     return A_hat
            
# using the indice strategy i am currently using, A_hat and A are both 1 dimensional. This might change.
def mse(A_hat, A):
    """
    Returns the mean squared error of estimate A_hat from ground truth A
    """
    return np.mean((A_hat - A)**2)

## Cross Validation for $\eta$

Next, we implement cross-validation for the $\eta$ radius parameter. Given a dataset, an error metric, and an estimation function, we evaluate a range of values of $\eta$. 

In [ ]:
# notes
    # we are only interested in the estimates that `estimate` makes for the
    # validation set in particular. For any validation fold, we must only consider observed values (need a ground truth)
    # We mask over these values to estimate, then extract the estimates using the original mask to compute the error.

In [287]:
def cross_validate_nn(Z, M, k, eta, estimate, error_metric):
    """
    Performs k-fold cross validation on using dataset `Z`, mask `M`, parameters `k` and `eta`,
    estimation function `estimate`, and error `error_metric`
    """
    # find observed indices and create splits off of those
    obvs_inds_x, obvs_inds_y = np.nonzero(M)
    shuf_inds = np.arange(len(obvs_inds_x))
    np.random.shuffle(shuf_inds)
    shuf_obvs_inds = np.array((obvs_inds_x[shuf_inds], obvs_inds_y[shuf_inds]))
    folds = np.array_split(shuf_obvs_inds, k, axis = 1)
    tot_error = 0
    for i in range(k):
        folds_ind = tuple(folds[i])
        #print(folds_ind)
        #print(Z.shape)
        val_data = mean_collapse(Z[folds_ind], d = 1)
        est_mask = M.copy()
        # hide observed values in the validation set
        est_mask[folds_ind] = 0
        A_hat = estimate(Z, est_mask, eta, folds_ind)
        # extract validation points
        est_val = A_hat[folds_ind]
        # in this use case, A and A_hat are 1 dimensional
        tot_error += error_metric(est_val, val_data)
    # return average error over k folds
    return tot_error / k
 

### Cross validation for Li (2019) NN

We use k-fold cross validation to tune hyperparameter $\eta$ for the Li (2019) NN model implemented above. We show results using k = 5 and k = # observed outcomes (LOOCV). 

In [264]:
# first, compute dissims from a subsection of the data to get a sense of the range of distances
# should change later, but M_hs is Z and A_hs is M
from itertools import product
rnd_ind = np.random.choice(37, size = 10, replace = False)
all_combs = product(rnd_ind, rnd_ind)
M_hs_collapse = mean_collapse(M_hs)
for u, v in all_combs:
    print(dissim_row(M_hs_collapse, A_hs, u, v))

0.0
269.0297067901234
1015.9899691358024
857.9251543209876
820.383487654321
520.2369684499315
1636.0393518518515
1755.354938271605
574.4270833333334
1026.5192901234568
269.0297067901234
0.0
1113.0328703703706
3134.1646604938273
2594.7826646090534
4078.115162037037
5380.932355967078
3916.369268077601
1720.8805555555555
5356.809220679013
1015.9899691358024
1113.0328703703706
0.0
145.7673611111111
383.7131172839506
1545.1593364197533
4437.703800154321
977.8233024691359
775.3533950617284
210.38927469135803
857.9251543209876
3134.1646604938273
145.7673611111111
0.0
1200.6007716049387
2384.920138888889
736.4311728395062
618.8354552469136
845.3027777777778
1598.7121913580243
820.383487654321
2594.7826646090534
383.7131172839506
1200.6007716049387
0.0
2410.28225308642
2880.8533950617275
2683.4371141975307
1894.2239583333333
930.5910493827159
520.2369684499315
4078.115162037037
1545.1593364197533
2384.920138888889
2410.28225308642
0.0
1789.7707475994512
492.41635802469136
918.4479166666666
101.

In [288]:
# try using HeartSteps data with eta 10 to 500? 
# k = 5
eta_range = np.arange(10, 510, step = 10)

min_error = float("inf")
best_eta = 0
for eta in eta_range:
    print(type(eta))
    error = cross_validate_nn(M_hs, A_hs, 5, eta, estimate_li_nn_inds, mse)
    print("Error for eta " + str(eta) + ": " + str(error))
    if error < min_error:
        min_error = error
        best_eta = eta


<class 'numpy.int32'>
Error for eta 10: 4458.23050515267
<class 'numpy.int32'>
Error for eta 20: 4407.896038065844
<class 'numpy.int32'>
Error for eta 30: 4442.295600137174
<class 'numpy.int32'>
Error for eta 40: 4650.209844730822
<class 'numpy.int32'>
Error for eta 50: 4468.966666283103
<class 'numpy.int32'>
Error for eta 60: 4475.949462484388
<class 'numpy.int32'>
Error for eta 70: 4415.347145553418
<class 'numpy.int32'>
Error for eta 80: 4453.767899883859
<class 'numpy.int32'>
Error for eta 90: 4790.137286588108
<class 'numpy.int32'>
Error for eta 100: 4616.856719233692
<class 'numpy.int32'>
Error for eta 110: 4303.62092863806
<class 'numpy.int32'>
Error for eta 120: 4329.9202555960555
<class 'numpy.int32'>
Error for eta 130: 4521.055450771797
<class 'numpy.int32'>
Error for eta 140: 4610.084645241007
<class 'numpy.int32'>
Error for eta 150: 4360.419040944247
<class 'numpy.int32'>
Error for eta 160: 4351.49066696151
<class 'numpy.int32'>
Error for eta 170: 4790.258071050569
<class '

In [290]:
print(best_eta)

470
